In [18]:
from langgraph.graph import StateGraph,START,END
from typing import  TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver#memory in ram 

In [5]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]#all Message type like AImessage,HumanMessage,SystemMessage,ToolMessage inherit from BaseMessage,
    #aur hame reducer function add krna padega q ki ye purani chz hata deta and nayi chat arakh leta hai .


In [6]:
llm=ChatOpenAI()

In [7]:
def chat_node(state:ChatState):
    #take user query from state
    messages=state['messages']
    #send it to llm 
    response=llm.invoke(messages)

    #store response in state
    return {'messages':[response]}

In [19]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)
#add nodes
graph.add_node('chat_node',chat_node)

#add edge
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

workflow=graph.compile(checkpointer=checkpointer)


In [13]:
initial_state={
    'messages':[HumanMessage(content='What is the capital of France?')]
}
workflow.invoke(initial_state)['messages']

[HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='65f32fcb-761c-4544-95a7-a9ce134e5616'),
 AIMessage(content='Paris', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 14, 'total_tokens': 15, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DZXVkXhNj91NFmsDB6uFhHsMgkZdv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019dd312-0779-7ab1-95ba-05ce05c48f28-0', usage_metadata={'input_tokens': 14, 'output_tokens': 1, 'total_tokens': 15, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

In [20]:
thread_id='1'#rahul ka jo interaction ho raha 1 thread shiv ka jo ho raha wo one thread
while True:
    user_msg=input('Type Here:')
    print('User:',user_msg)
    if user_msg.strip().lower() in ['exit','quit','bye']:
        break
    config={'configurable':{'thread_id':thread_id}}
    response=workflow.invoke({'messages':[HumanMessage(content=user_msg)]},config=config)
    print('AI',response['messages'][-1].content)

User: hi  my name is shiv
AI Hello Shiv! How can I assist you today?
User: what is my name?
AI Your name is Shiv.
User: can you ad 10 to 100
AI Sure! 10 + 100 = 110.
User: can yo multiply results with 2
AI Certainly! 110 multiplied by 2 is 220.
User: quit


#above user has no memory so that it will not remember the previous message.

In [22]:
workflow.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='hi  my name is shiv', additional_kwargs={}, response_metadata={}, id='9ed32a2e-f700-4b50-8732-8e9900ba9c3d'), AIMessage(content='Hello Shiv! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DZZW3jZK4RuaBlIpcJarM78ODp0HY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019dd387-a2a3-7f23-b141-75a40c29b00f-0', usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMes